# Explore CERN Open Data record 69747

This notebook opens a NanoAOD ROOT file directly from CERN Open Data through XRootD and reads selected columns as Awkward Arrays. Uproot requests only the remote data needed for each operation, so the complete 133 MB file does not need to be downloaded.

In [1]:
# @title Setup: install missing packages and configure the remote ROOT file
import importlib.util
import subprocess
import sys

required_packages = {
    "awkward": "awkward==2.12.0",
    "uproot": "uproot==5.7.5",
    "XRootD": "xrootd==6.1.0",
    "fsspec_xrootd": "fsspec-xrootd==0.5.5",
    "vector": "vector==1.8.1",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import awkward as ak
import uproot

file_url = (
    "root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/"
    "WJetsToLNu_TuneCP5_13TeV-madgraphMLM-pythia8/NANOAODSIM/"
    "106X_mcRun2_asymptotic_v17-v1/270000/"
    "00702195-E707-3743-8BBA-57EB9DEE1DBA.root"
)

print(f"Uproot {uproot.__version__}")
print(f"Awkward {ak.__version__}")
print("Input: CERN Open Data via XRootD")
print(f"URL: {file_url}")

Uproot 5.7.5
Awkward 2.12.0
Input: CERN Open Data via XRootD
URL: root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/WJetsToLNu_TuneCP5_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v1/270000/00702195-E707-3743-8BBA-57EB9DEE1DBA.root


## Inspect the file

In [2]:
root_file = uproot.open(file_url)
root_file.classnames()

{'tag;1': 'TObjString',
 'Events;1': 'TTree',
 'LuminosityBlocks;1': 'TTree',
 'Runs;1': 'TTree',
 'MetaData;1': 'TTree',
 'ParameterSets;1': 'TTree'}

In [3]:
events = root_file["Events"]
print(f"Events: {events.num_entries:,}")
print(f"Branches: {len(events.keys()):,}")
events.keys()[:30]

Events: 133,692
Branches: 1,504


['run',
 'luminosityBlock',
 'event',
 'HTXS_Higgs_pt',
 'HTXS_Higgs_y',
 'HTXS_stage1_1_cat_pTjet25GeV',
 'HTXS_stage1_1_cat_pTjet30GeV',
 'HTXS_stage1_1_fine_cat_pTjet25GeV',
 'HTXS_stage1_1_fine_cat_pTjet30GeV',
 'HTXS_stage1_2_cat_pTjet25GeV',
 'HTXS_stage1_2_cat_pTjet30GeV',
 'HTXS_stage1_2_fine_cat_pTjet25GeV',
 'HTXS_stage1_2_fine_cat_pTjet30GeV',
 'HTXS_stage_0',
 'HTXS_stage_1_pTjet25',
 'HTXS_stage_1_pTjet30',
 'HTXS_njets25',
 'HTXS_njets30',
 'nboostedTau',
 'boostedTau_chargedIso',
 'boostedTau_eta',
 'boostedTau_leadTkDeltaEta',
 'boostedTau_leadTkDeltaPhi',
 'boostedTau_leadTkPtOverTauPt',
 'boostedTau_mass',
 'boostedTau_neutralIso',
 'boostedTau_phi',
 'boostedTau_photonsOutsideSignalCone',
 'boostedTau_pt',
 'boostedTau_puCorr']

## Read a bounded sample

Selecting branches and limiting the entry range keeps the first exploration fast and memory-efficient.

In [4]:
branches = [
    "run",
    "luminosityBlock",
    "event",
    "MET_pt",
    "nMuon",
    "Muon_pt",
    "Muon_eta",
    "nElectron",
    "Electron_pt",
]

sample = events.arrays(branches, entry_stop=10_000, library="ak")
sample.type.show()

10000 * {
    run: uint32,
    luminosityBlock: uint32,
    event: uint64,
    MET_pt: float32,
    nMuon: uint32,
    Muon_pt: var * float32,
    Muon_eta: var * float32,
    nElectron: uint32,
    Electron_pt: var * float32
}


In [5]:
sample[:3]

<Array [{run: 1, ...}, ..., {run: 1, ...}] type='3 * {run: uint32, luminosi...'>

## Select events with two energetic muons

Jagged particle collections can have a different number of values in every event. Awkward applies the particle-level cut within each event, and `ak.num` counts the surviving particles.

In [6]:
energetic_muons = sample.Muon_pt > 20
event_mask = ak.num(sample.Muon_pt[energetic_muons], axis=1) >= 2
selected = sample[event_mask]

print(f"Selected {len(selected):,} of {len(sample):,} sampled events")
selected[["run", "event", "MET_pt", "Muon_pt"]][:5]

Selected 59 of 10,000 sampled events


<Array [{run: 1, event: 6053920, ...}, ...] type='5 * {run: uint32, event: ...'>

## Iterate over the complete tree in chunks

For full-dataset analysis, `uproot.iterate` bounds memory use. Remove the `break` after developing the per-chunk calculation.

In [7]:
for chunk in uproot.iterate(
    f"{file_url}:Events",
    filter_name=["MET_pt", "Muon_pt"],
    step_size="50 MB",
    library="ak",
):
    print(f"Read {len(chunk):,} events in this chunk")
    break

Read 133,692 events in this chunk
